# Bell examples (v0.10)

**Author:** Kolokolchikov Sergey Dmitrievich (`sergey.bell13@gmail.com`)

This notebook runs every numbered example of the Julia package Bell.
It is not a product release (`Project.toml` version `0.10.0`).

Open this file in Jupyter with the working directory `examples/`.
The first cell activates the examples environment (`examples/Project.toml`),
where Bell is a dev dependency on the repository root.

## What is an example, and what is not

| File | Role |
|------|------|
| `10`–`18` | Program examples. They load Bell and a structure file. |
| `common/print_util.jl` | Helper: `banner` / `section` for `10`–`18`. |
| `common/reference_values.jl` | Helper: OptiM path and COSY numbers for comparison tables. |

Each example is `run_example("NN_name/run.jl")` in a fresh module, so
constants of different examples do not clash. The printed output is the
same as `julia --project=examples examples/NN_name/run.jl`.

## Blocks of the program vs examples

| `src/` block | Example that uses it |
|--------------|----------------------|
| `coordinates.jl` | `15` (main), also `10`, `14` |
| `elements.jl` | `10`, `17` |
| `structure.jl` | `10`–`16`, `18`; FODO cell below (no OptiM) |
| `hamiltonians.jl` | `11`, `17` |
| `lie_algebra.jl` | `11`, `17` |
| `differential_algebra.jl` | `11`, `12` (check only) |
| `linear.jl` | `10` |
| `higher_order.jl` | `12` |
| `normal_form.jl` | `18` |
| `tracking.jl` | `11` |
| `exact_integrator.jl` | `12`, `14` |
| `longitudinal.jl` | `12`, `14` |
| `acceleration.jl` | `14` |
| `optimizers.jl` | `13` |
| `plotting.jl` | `10`, `12`, `13`, `14` |
| `symplectic.jl` | `10`, `11`, `15` |
| `optim_converter.jl` | `10` and every script that calls `convert_optim` |

`16` compares an external BLonD `(dt, dE)` drift with Bell tracking.
Those BLonD formulae are not a `src/` block.


In [1]:
import Pkg
# Jupyter cwd is the notebook folder (examples/).
Pkg.activate(pwd())
Pkg.instantiate()
ENV["GKSwstype"] = "100"   # headless Plots
using Dates

# Each example file is a script for Main.  A fresh module avoids const
# clashes between examples but must still have include().
function run_example(path)
    m = Module()
    Core.eval(m, :(include(x) = Base.include(@__MODULE__, x)))
    Base.include(m, path)
    return nothing
end

println("cwd      = ", pwd())
println("project  = ", Base.active_project())
println("started  = ", now())


  Activating project at `C:\GIT_REPOS\BELL\examples`


cwd      = C:\GIT_REPOS\BELL\examples
project  = C:\GIT_REPOS\BELL\examples\Project.toml
started  = 

2026-10-30T12:08:42.652


## FODO through Bell (`lattices/fodo.bell`)

This cell loads a structure file directly:
`load_structure` → `Lattice` → one-turn `R` → closed Twiss.
It is the structure-file block without OptiM.


In [2]:
using Bell, Printf
lat = load_structure(joinpath(pwd(), "lattices", "fodo.bell"))
C = circumference(lat)
po = periodic_optics(ring_matrix(lat), lat.ref, C)
@printf("  name      = %s\n", lat.name)
@printf("  elements  = %d\n", length(lat.elements))
@printf("  C         = %.6f m\n", C)
@printf("  Qx        = %.6f\n", po.x.mu / 2pi)
@printf("  Qy        = %.6f\n", po.y.mu / 2pi)
@printf("  alpha_c   = %.8f\n", po.alpha_c)
@printf("  CLOSED check uses linear.jl + structure.jl + hamiltonians.jl\n")


[ Info: Precompiling IJuliaExt [6e0e26ac-fb43-584e-b323-fdbabcf95812] (caches not reused: 2 for different dependency version already loaded)
Precompiling packages...


   1.5 s  ✓ IJuliaExt (serial)
  1 dependency successfully precompiled in 2 seconds


  name      = fodo
  elements  = 128
  C         = 188.800000 m
  Qx        = 0.955869
  Qy        = 0.798840
  alpha_c   = 0.03888109


  CLOSED check uses linear.jl + structure.jl + hamiltonians.jl


## 10 — Linear optics of `magnetic_2`

OptiM `.opt` is converted to `lattices/magnetic_2.bell`. Bell loads the
structure file. Closed Twiss, dispersion, tunes, comparison with OptiM
and COSY numbers.

Blocks: converter, structure, coordinates, elements, Hamiltonians,
linear, symplectic, plotting.


In [3]:
run_example(joinpath("10_linear_optics", "run.jl"))



  Linear optics of magnetic_2


--- 1. Lattice file



  file                 : C:\GIT_REPOS\BELL\examples\common\..\lattices\magnetic_2.opt
  header               : Energy = 270.0 MeV (kinetic), Mass = 1876.544 MeV
  sequence tokens      : 84, 'Number of periods' in file = 2, repetitions used = 1

--- 2. Reference particle

particle            : deuteron (Z = 1.0)
rest energy used    : 1875.61294257 MeV
kinetic energy      : 270.0 MeV
gamma0, beta0       : 1.14395294, 0.48563473
p0*c                : 1041.98416 MeV
B*rho               : 3.475685 T*m
1/gamma0^2          : 0.76415891
  (physical deuteron mass would be 1875.61294257 MeV -> B*rho = 3.4756850 T*m; not used)

--- 3. Elements



  OS2   x8   Drift(OS2, L=0.06 m)
  QF    x16  Quadrupole(QF, L=0.21 m, k1=0.5213648481160231 1/m^2)
  BM    x16  SBend(BM, L=5.329414799999999 m, angle=0.3927899887227164 rad, k1=0.0)
  QD    x16  Quadrupole(QD, L=0.21 m, k1=-0.5807200537704216 1/m^2)
  OS1   x6   Drift(OS1, L=0.12 m)
  SF2   x4   Sextupole(SF2, L=0.12 m, k2=0.0 1/m^3)
  SD    x4   Sextupole(SD, L=0.12 m, k2=-0.0 1/m^3)
  SF1   x2   Sextupole(SF1, L=0.12 m, k2=0.0 1/m^3)
  QF1   x4   Quadrupole(QF1, L=0.21 m, k1=0.556724784009991 1/m^2)
  OBM   x4   Drift(OBM, L=5.329414799999999 m)
  QD1   x4   Quadrupole(QD1, L=0.21 m, k1=-0.556724784009991 1/m^2)

  number of elements    : 84
  circumference C       : 117.3882960 m
  total bending angle   : 360.083337421 deg


  dipoles: 16, quads: 40, sextupoles: 10

--- 4. Element linear maps

  R = exp(L S A), A = Hessian of H at z=0; symplectic error |R'SR - S|.



  OS2    R11= 1.000000 R12= 0.060000 R21= 0.000000 R33= 1.000000 R16= 0.000000 R56= 0.045850  sympl.err=0.0e+00
  QF     R11= 0.988526 R12= 0.209196 R21=-0.109068 R33= 1.011518 R16= 0.000000 R56= 0.160473  sympl.err=1.2e-17
  BM     R11= 0.923845 R12= 5.193428 R21=-0.028211 R33= 1.000000 R16= 1.033282 R56= 3.936533  sympl.err=4.4e-16
  QD     R11= 1.012832 R12= 0.210897 R21= 0.122472 R33= 0.987222 R16= 0.000000 R56= 0.160473  sympl.err=2.2e-16
  OS1    R11= 1.000000 R12= 0.120000 R21= 0.000000 R33= 1.000000 R16= 0.000000 R56= 0.091699  sympl.err=0.0e+00


  SF2    R11= 1.000000 R12= 0.120000 R21= 0.000000 R33= 1.000000 R16= 0.000000 R56= 0.091699  sympl.err=0.0e+00
  SD     R11= 1.000000 R12= 0.120000 R21= 0.000000 R33= 1.000000 R16= 0.000000 R56= 0.091699  sympl.err=0.0e+00
  SF1    R11= 1.000000 R12= 0.120000 R21= 0.000000 R33= 1.000000 R16= 0.000000 R56= 0.091699  sympl.err=0.0e+00
  QF1    R11= 0.987749 R12= 0.209142 R21=-0.116434 R33= 1.012301 R16= 0.000000 R56= 0.160473  sympl.err=2.2e-16
  OBM    R11= 1.000000 R12= 5.329415 R21= 0.000000 R33= 1.000000 R16= 0.000000 R56= 4.072520  sympl.err=4.4e-16
  QD1    R11= 1.012301 R12= 0.210860 R21= 0.117391 R33= 0.987749 R16= 0.000000 R56= 0.160473  sympl.err=2.2e-16

--- 5. One-turn matrix and closed optics

  one-turn 6x6 matrix (canonical x, px, y, py, l, delta):
    -9.723937e-01  4.527066e+00  0.000000e+00  0.000000e+00  0.000000e+00  3.882477e-02
    -1.206847e-02 -9.722043e-01  0.000000e+00  0.000000e+00  0.000000e+00 -1.388817e-03
     0.000000e+00  0.000000e+00 -9.717407e-01  9.57

... (2 outputs left out)

  normal-form check: |A^-1 R A - rotation| = 2.87e-14

--- 6. Optics along the ring + CLOSED check



  rel. beta_x 2.1e-14, alpha_x 3.2e-14, rel. beta_y 7.6e-14, alpha_y 9.3e-15, D 1.7e-12, D' 3.3e-14, Qfrac 3.9e-15/3.8e-15
  CLOSED: yes (all closure errors < 1e-8)

  tunes Qx = 2.46245165   Qy = 2.46201624  (integer from accumulated phase advance)
  max beta_x = 19.3681 m, max beta_y = 19.3620 m, max D = 11.6925 m, min D = 0.0109 m

  alpha_c from (1/C) int h*D ds (trapezoid)  = 0.23039883  (independent check)

--- 7. Plot



  written: C:\GIT_REPOS\BELL\examples\10_linear_optics\output\magnetic_2_twiss.png

--- 8. Comparison with OptiM / COSY

  quantity                         Bell          reference         diff  source
  beta_x(s=0) [m]           19.36789516            19.3675    3.952e-04  OptiM Initial


  beta_y(s=0) [m]           4.052354283            4.05447   -2.116e-03  OptiM Initial
  alpha_x(s=0)          -0.000405142371       -0.000251909   -1.532e-04  OptiM Initial
  alpha_y(s=0)         -0.0003588576594       -0.000188151   -1.707e-04  OptiM Initial
  D_x(s=0) [m]            0.01781755944                  0    1.782e-02  OptiM Initial
  D'_x(s=0)            -0.0008132259962        -0.00050616   -3.071e-04  OptiM Initial
  Qx fractional            0.4624516526       0.4462838512    1.617e-02  COSY wp_natural.dat
  Qy fractional            0.4620162437        0.476682443   -1.467e-02  COSY wp_natural.dat
  alpha_c                  0.2303987449       0.2303626035    3.614e-05  COSY integrals_Icirc.dat

Done.


## 11 — Lie maps and tracking

Element `H`, Dragt–Finn `f3`, ring Lie product versus the independent
RK4 Taylor map, chromaticity, symplectic factor-by-factor tracking.

Blocks: Hamiltonians, Lie algebra, differential algebra (check),
tracking, symplectic.


In [4]:
run_example(joinpath("11_lie_maps", "run.jl"))



  2nd-order Lie maps, symplectic tracking, Taylor-map check

  test amplitude z_amp (x, px, y, py, l, delta) = 

[0.003, 0.0002, 0.002, 0.0002, 0.001, 0.002]

--- 1. Elements: H, Lie map, f3, symplecticity

  Lie map = exp(-L:H:) z to order 2; f3 = Dragt-Finn cubic generator.

  elem  |lin-expLSA|  recon.err   TM sym@0 TM sym@amp LF sym@amp  nterms f3


  OS2      6.9e-18    0.0e+00    0.0e+00    0.0e+00    0.0e+00          3
  QF       2.2e-16    5.6e-17    4.4e-16    1.6e-12    6.7e-16          7


  BM       1.8e-15    8.9e-16    4.4e-16    2.5e-09    4.4e-16         13
  QD       2.2e-16    2.8e-17    2.2e-16    1.9e-12    3.3e-16          7
  OS1      1.4e-17    0.0e+00    0.0e+00    0.0e+00    0.0e+00          3
  SF2      1.4e-17    0.0e+00    0.0e+00    0.0e+00    0.0e+00          3


  SD       1.4e-17    0.0e+00    0.0e+00    0.0e+00    0.0e+00          3
  SF1      1.4e-17    0.0e+00    0.0e+00    0.0e+00    0.0e+00          3
  QF1      1.1e-16    4.2e-17    2.2e-16    1.8e-12    4.4e-16          7
  OBM      8.9e-16    0.0e+00    0.0e+00    0.0e+00    0.0e+00          3
  QD1      1.1e-16    4.2e-17    2.2e-16    1.8e-12    4.4e-16          7



  QF: H = e(delta) - ps + k1/2 (x^2 - y^2)
    H (order<=4) = 0.26068*x^2 + 0.5*px^2 - 0.26068*y^2 + 0.5*py^2 + 0.38208*delta^2 - 0.5*px^2*delta - 0.5*py^2*delta - 0.09011*delta^3 + 0.125*px^4 + 0.25*px^2*py^2 + 0.5*px^2*delta^2 + 0.125*py^4 + 0.5*py^2*delta^2 + 0.0040371*delta^4
    f3           = 0.00041763*x^2*delta - 0.011408*x*px*delta + 0.1042*px^2*delta + 0.00042149*y^2*delta + 0.011584*y*py*delta + 0.10581*py^2*delta + 0.018923*delta^3

  BM: H = e(delta) - (1+hx) ps + h x + h^2 x^2/2 + k1/2 (x^2-y^2)
    H (order<=4) = 0.002716*x^2 - 0.073702*x*delta + 0.5*px^2 + 0.5*py^2 + 0.38208*delta^2 + 0.036851*x*px^2 + 0.036851*x*py^2 - 0.5*px^2*delta - 0.5*py^2*delta - 0.09011*delta^3 - 0.036851*x*px^2*delta - 0.036851*x*py^2*delta + 0.125*px^4 + 0.25*px^2*py^2 + 0.5*px^2*delta^2 + 0.125*py^4 + 0.5*py^2*delta^2 + 0.0040371*delta^4
    f3           = -5.0771e-5*x^3 + 0.0049879*x^2*px + 0.0020666*x^2*delta - 0.16334*x*px^2 - 0.13535*x*px*delta - 0.19138*x*py^2 - 0.02804*x*delta^2 - 0.47

... (4 outputs left out)

  one turn: max |Lie tracking - Taylor map| = 4.72e-10 (expected O(|z|^3))


  symplectic error of 1-turn tracking Jacobian at z0: 2.18e-14 (Taylor map: 1.28e-06)



  1000 turns in 6.4 s; action Jx: mean 1.0258e-07, rel. spread 2.37e-03; Jy: mean 1.2324e-07, rel. spread 2.71e-03
  (spread is due to the nonlinear (2nd-order) terms and d-dependence of Twiss, not to loss of symplecticity)
  final delta = 1.000000000000000e-03 (must equal start 1.000000000000000e-03: magnetic lattice)

--- 5. Comparison with COSY output numbers

  quantity               Bell           COSY         diff  source


  xi_x            -1.46244729    -1.84080073    3.784e-01  COSY\src\dat\magnetic_2\chrom_check_I0_FR0.dat chrom_x_DA (all SG=0)
  xi_y            -2.22668720    -1.84767883   -3.790e-01  COSY\src\dat\magnetic_2\chrom_check_I0_FR0.dat chrom_y_DA (all SG=0)
  alpha_0          0.23039874     0.23036260    3.614e-05  COSY\src\dat\magnetic_2\integrals_Icirc.dat / sct_study audit.json alpha0_mean

Done.


## 12 — Higher order (`D2`, `η`, `ξ`)

Second-order dispersion, momentum compaction, slip, chromaticity.
Cross-check with the exact-H integrator.

Blocks: higher order, differential algebra, exact integrator,
longitudinal (tune formula), plotting.


In [5]:
run_example(joinpath("12_higher_order", "run.jl"))



  Second-order dispersion, compaction, slip, chromaticity

  lattice magnetic_2, C = 117.388296 m, E_kin = 270.0 MeV, gamma0 = 1.14395294, beta0 = 0.48563473

--- 1. One-turn Taylor map (order 3)

  Element maps by RK4 in truncated polynomials, composed.



  built in 0.0 s

--- 2. Second-order dispersion

  (I-R4) D2 = Q4(D1,0,1): delta^2 part of the closed orbit.

  s=0:  D =  0.01781756 m   D' = -8.13225974e-04   D2 = -10.01667514 m   D2' =  4.57680678e-01
  along ring (element exits): max D = 11.69251 m, max|D2| = 14.36003 m, D2 range [-10.04418, 14.36003] m


  closure: D2(C) - D2(0) = 6.07e-09, D2'(C) - D2'(0) = 2.11e-10



  vs RK4 element maps: max |D2 Lie minus RK4| = 6.07e-09

--- 3. Momentum compaction and slip factor to 2nd order

  dC/C = alpha0 d + alpha1 d^2 ;  dT/T0 = eta0 d + eta1 d^2.

  l1 = (l|d)_closed = 62.65719627 m,  l2 = (l|d^2)_closed = -71.47518662 m
  v/v0 - 1 = 0.76415891 d + -0.27033011 d^2  (check vs exact at d=1e-3: 1.62e-11)

  alpha0 = 0.23039874   alpha1 = 0.33854820   gamma_t = 2.083339
  eta0   = -0.53376017   eta1   = 1.01675589   (identity formula: 1.01675589, diff 0.0e+00)
  delta_t (2nd fixed point of longitudinal motion, alpha-bucket) ~ -eta0/eta1 = 0.5250



  observables (Lie factor): eta0 = -0.53376017  eta1 = 1.01675589  (Taylor map eta0 = -0.53376017, eta1 = 1.01675589)
  observables Twiss from F.R: Qx = 0.46245165  Qy = 0.46201624  alpha_c = 0.23039874

--- 4. Cross-check by exact-H tracking

  Closed orbit at fixed delta (Newton), dC/C from l, quadratic fit.

  fit x_co = a d + b d^2 + c d^3:    a = 0.01780844 (map D 0.01781756),  b = -10.016611 (map D2 -10.016675)


  fit dC/C = a d + b d^2 + c d^3:    a = 0.23039897 (map alpha0 0.23039874), b = 0.338547 (map alpha1 0.338548)

--- 5. Chromaticity, first and second order

  Map: Q(d) from the Jacobian on the off-momentum fixed point.
  Tracking: exact H, 256 turns of 1e-5 m oscillation, linear-tune fit.

  map:  Qx0 = 0.46245165  xi1_x = -1.462449  xi2_x = d2Qx/dd2 = 8.5534
        Qy0 = 0.46201624  xi1_y = -2.226690  xi2_y = d2Qy/dd2 = 8.1072



  tracking fit: Qx0 = 0.46245055  xi1_x = -1.462443  xi2_x = 8.5532
                Qy0 = 0.46201458  xi1_y = -2.226681  xi2_y = 8.1069

  delta         Qx track        Qx map   Qx map-poly      Qy track        Qy map


  -1.5e-03   0.464653847   0.464655605   0.464654949   0.465363745   0.465366522
  -1.0e-03   0.463917269   0.463918568   0.463918378   0.464245321   0.464247308
  -5.0e-04   0.463182836   0.463183969   0.463183946   0.463128935   0.463130640
  0.0e+00    0.462450545   0.462451653   0.462451653   0.462014580   0.462016244
  5.0e-04    0.461720392   0.461721476   0.461721497   0.460902252   0.460903876
  1.0e-03    0.460992374   0.460993305   0.460993480   0.459791946   0.459793320
  1.5e-03    0.460266488   0.460267019   0.460267601   0.458683656   0.458684381



  written: C:\GIT_REPOS\BELL\examples\12_higher_order\output\chromaticity_natural.png

--- 6. Longitudinal motion from eta0, eta1

turn-by-turn longitudinal motion (n = turn number, phi = -h w0 dt, delta = dp/p):
  dphi/dn   = -2 pi h (eta0 delta + eta1 delta^2)
  ddelta/dn = (Z V T/(beta^2 E)) (sin(phi_s + phi) - sin(phi_s))
  H(phi,delta) = -2 pi h (eta0 delta^2/2 + eta1 delta^3/3)
                 + (Z V T/(beta^2 E)) (cos(phi_s + phi) + phi sin(phi_s))
  small amplitude:  Q_s = sqrt( h eta0 Z V T cos(phi_s) / (2 pi beta^2 E) ),
  stable iff eta0*cos(phi_s) > 0.  eta1 adds a second fixed point at
  delta ~ -eta0/eta1 (alpha-bucket) and makes the bucket asymmetric.
  example at 270 MeV, h=5, V=0.2 MV, T=1, phi_s=180 deg: Q_s = 0.012957

--- 7. Exact integrator vs Lie tracking vs linear map (one turn, no RF)



  |exact - Lie(order 3)| = 2.36e-08,  |exact - Taylor map| = 2.35e-08,  |exact - linear| = 6.90e-05


  symplectic error of exact-integrator one-turn Jacobian at z0: 1.35e-14

--- 8. Notes

  COSY comparison items (from phase 1) remain open:
  tunes differ from COSY by ~0.015; natural chromaticity differs (COSY -1.84/-1.85);
  sextupole strengths in magnetic_2.opt are all zero (natural lattice above).

  COSY natural chromaticity (reference only): xi_x = -1.840801, xi_y = -1.847679

Done.


## 13 — Optional sextupole families

Places `SXD` / `SXB` / `SYB` and solves two example targets. Other
examples leave sextupoles at zero.

Block: optimizers (optional; not part of the model).


In [6]:
run_example(joinpath("13_sextupole_optimization", "run.jl"))



  Sextupole families and example optimisation


--- 1. Placement

  Maxima of D_x, beta_x, beta_y with D_x > 0.5 m.



  fam  max of      s_max     value slot     s_slot   beta_x   beta_y      D_x


  SXD  D_x        23.418   11.6925 SF1      23.478   19.368    4.053   11.693
  SXD  D_x        82.112   11.6925 SF1      82.172   19.368    4.053   11.693
  SXB  beta_x     11.679   16.4600 SF2      11.739   16.460    3.323    5.828
  SXB  beta_x     35.156   16.4600 SF2      35.216   16.460    3.323    5.868
  SXB  beta_x     70.373   16.4600 SF2      70.433   16.460    3.323    5.828
  SXB  beta_x     93.851   16.4600 SF2      93.911   16.460    3.323    5.868
  SYB  beta_y      5.799   18.7886 OS1       5.869    3.661   18.789    1.123
  SYB  beta_y     17.678   18.7819 SD       17.608    3.660   18.782    4.922
  SYB  beta_y     29.277   18.7819 SD       29.347    3.660   18.782    4.922
  SYB  beta_y     41.156   18.7886 OS1      41.086    3.661   18.789    1.123
  SYB  beta_y     64.494   18.7886 OS1      64.564    3.661   18.789    1.123
  SYB  beta_y     76.372   18.7819 SD       76.302    3.660   18.782    4.922
  SYB  beta_y     87.971   18.7819 SD       88.041    3.660   18


  family         dxi1_x       dxi1_y      dalpha1        deta1   dD2(0) [m]
  SXD           4.32505     -0.90500     -1.63410     -1.63410     89.58832
  SXB           3.67670     -0.74213     -0.40887     -0.40887     -0.05541
  SYB           0.84228     -4.33448     -0.24173     -0.24173      9.41588



--- 3. Solutions

  A: xi1_x = xi1_y = alpha1 = 0
  B: xi1_x = xi1_y = 0, min rms D2

  solution A: 

SXD: k2 = 0.230537 m^-3 (k2*L = 0.027664 m^-2, B'' = 0.80127 T/m^2);  SXB: k2 = 0.265705 m^-3 (k2*L = 0.031885 m^-2, B'' = 0.92351 T/m^2);  SYB: k2 = -0.607342 m^-3 (k2*L = -0.072881 m^-2, B'' = -2.1109 T/m^2)
  solution B: SXD: k2 = 0.145525 m^-3 (k2*L = 0.017463 m^-2, B'' = 0.5058 T/m^2);  SXB: k2 = 0.365558 m^-3 (k2*L = 0.043867 m^-2, B'' = 1.2706 T/m^2);  SYB: k2 = -0.606689 m^-3 (k2*L = -0.072803 m^-2, B'' = -2.1087 T/m^2)



--- 4. Before / after

  All from the order-3 one-turn map; xi2 = d2Q/ddelta2.

  quantity              natural  A (xi,alpha1)   B (xi,D2rms)
  Qx frac              0.462452       0.462452       0.462452


  Qy frac              0.462016       0.462016       0.462016
  xi1_x               -1.462449       0.000005       0.000003
  xi1_y               -2.226690      -0.000007      -0.000007
  xi2_x                8.553412       3.930988       5.165654
  xi2_y                8.107215      -0.710743       1.242211
  D(0) [m]             0.017818       0.017818       0.017818
  D2(0) [m]          -10.016675       4.903346      -2.712116
  max|D2| [m]         14.360032      10.018958       3.790496
  rms D2 [m]           7.532204       5.720554       2.403490
  alpha0               0.230399       0.230399       0.230399
  alpha1               0.338548       0.000000       0.097933
  eta0                -0.533760      -0.533760      -0.533760
  eta1                 1.016756       0.678208       0.776141

  (linear-model prediction check A: xi1_x, xi1_y, alpha1 should be ~0)

--- 5. Tracking check of solution A



  tracking fit A: xi1_x = 0.00001  xi2_x = 3.931   xi1_y = 0.00000  xi2_y = -0.711
  map         A: xi1_x = 0.00001  xi2_x = 3.931   xi1_y = -0.00001  xi2_y = -0.711



  written: C:\GIT_REPOS\BELL\examples\13_sextupole_optimization\output\chromaticity_corrected.png
  written: C:\GIT_REPOS\BELL\examples\13_sextupole_optimization\output\sextupoles_dispersion2.png

Done.


## 14 — Finite RF cavity and acceleration

Cavity checks, synchrotron tune, ramp 10 → 270 MeV, adiabatic damping.
Injection energy 10 MeV is assumed (not in the OptiM file). Takes a
few minutes.

Blocks: exact integrator, acceleration, longitudinal, plotting.


In [7]:
run_example(joinpath("14_rf_acceleration", "run.jl"))



  Finite RF cavity, synchrotron motion, acceleration


--- 1. Parameters

  injection E_kin = 10.0 MeV (ASSUMED, 5 MeV/u; not given in magnetic_2.opt)
  final     E_kin = 270.0 MeV (135 MeV/u)

  cavity: L = 1.00 m in the first OBM drift (element 37)


          V = 0.200 MV, h = 5, phi_s = 150.0 deg
  C = 117.388296 m (unchanged)
  transit factor T = sin(th/2)/(th/2), th = 2 pi h L/C = 0.26762 rad -> T = 0.997018



  injection: beta = 0.102852 gamma = 1.005332 p0 = 193.9388 MeV/c Brho = 0.64691 T m  f0 = 262.6685 kHz  f_RF = 1.3133 MHz
  final    : beta = 0.485635 gamma = 1.143953 p0 = 1041.9842 MeV/c Brho = 3.47569 T m  f0 = 1240.2397 kHz  f_RF = 6.2012 MHz

  energy gain per turn Z V T sin(phi_s) = 0.099702 MeV -> about 2608 turns for the ramp

--- 2. Cavity checks

  Single passes through the cavity at 270 MeV (no renormalisation).



  synchronous particle: delta_out = 1.9701321011e-04
                       dE = 0.099700857 MeV   vs Z V T sin(phi_s) = 0.099701840 MeV

  thin limit L = 0.1    m: delta_out = 3.802214653167e-04  (thin kick 3.802277730094e-04, diff -6.31e-09)
  thin limit L = 0.01   m: delta_out = 3.802276257953e-04  (thin kick 3.802277730094e-04, diff -1.47e-10)
  thin limit L = 0.001  m: delta_out = 3.802277631235e-04  (thin kick 3.802277730094e-04, diff -9.89e-12)



  symplectic error of cavity map Jacobian: 2.22e-16

--- 3. Synchrotron tune at fixed energy (phi_s = 180 deg)

  Q_s = sqrt(h eta0 Z V T cos(phi_s) / (2 pi beta^2 E)), eta0 = alpha0 - 1/gamma^2.



  E =   10.0 MeV: alpha_c = 0.23039874 eta0 = -0.75902275
                 Q_s tracking = 0.0784688  smooth = 0.0777053 (diff 9.83e-03)  discrete-kick = 0.0784986 (diff -3.79e-04)

  E =  270.0 MeV: alpha_c = 0.23039874 eta0 = -0.53376017


... (7 outputs left out)

  (a) C:\GIT_REPOS\BELL\examples\14_rf_acceleration\output\longitudinal_ramp.gif
      every 20th turn; left: absolute, right: delta / current bucket height


[ Info: Saved animation to C:\GIT_REPOS\BELL\examples\14_rf_acceleration\output\longitudinal_ramp.gif


  (b) C:\GIT_REPOS\BELL\examples\14_rf_acceleration\output\longitudinal_elements.gif
      one turn at injection, frame = exit of each element

Done.


[ Info: Saved animation to C:\GIT_REPOS\BELL\examples\14_rf_acceleration\output\longitudinal_elements.gif


## 15 — User ↔ canonical coordinates

Exact nonlinear conversion `(x, x', y, y', φ, δ) ↔ (x, px, y, py, l, δ)`.
The user Jacobian is not symplectic.

Block: coordinates.


In [8]:
run_example(joinpath("15_coordinates", "run.jl"))



  Exact coordinate transformation user <-> canonical


--- 1. Formulas

  canonical -> user: x' = px/ps, y' = py/ps, ps = sqrt((1+delta)^2 - px^2 - py^2)
                     phi = 2 pi h l / (C v(delta)/v0)
  user -> canonical: px = x'(1+delta)/sqrt(1+x'^2+y'^2), py likewise,
                     l = phi C (v/v0) / (2 pi h)
  Both functions are generic: on TaylorN they give the full expansion.

  x'(px,py,delta) to order 3 : 1.0*px - 1.0*px*delta + 0.5*px^3 + 0.5*px*py^2 + 1.0*px*delta^2
  phi(l,delta)    to order 3 : 0.267624*l - 0.204507*l*delta + 0.228623*l*delta^2

  largest nonlinear coefficient in T: 1.500e+00  (> 0 => conversion is nonlinear)

--- 2. Polynomial round trip T(T^-1(u)) - u

  component 1: max|coef| orders <= 3: 0.00e+00   orders > 3 (kept up to JET_ORDER): 0.00e+00


  component 2: max|coef| orders <= 3: 0.00e+00   orders > 3 (kept up to JET_ORDER): 0.00e+00
  component 3: max|coef| orders <= 3: 0.00e+00   orders > 3 (kept up to JET_ORDER): 0.00e+00
  component 4: max|coef| orders <= 3: 0.00e+00   orders > 3 (kept up to JET_ORDER): 0.00e+00
  component 5: max|coef| orders <= 3: 1.11e-16   orders > 3 (kept up to JET_ORDER): 0.00e+00
  component 6: max|coef| orders <= 3: 0.00e+00   orders > 3 (kept up to JET_ORDER): 0.00e+00

--- 3. Point residuals at large amplitude

  u = (5 mm, 20 mrad, 5 mm, 20 mrad, 1.0 rad, 1e-2)

  exact functions round trip     : max|u - T(T^-1 u)| = 0.00e+00
  polynomial T (order 3) vs exact T at that point: max diff = 8.73e-07 (truncation error, O(amp^4))


  compare: linearised T would give max diff = 7.61e-03

--- 4. One-turn map coefficients in both systems (270 MeV, RF off)

  coefficient                             canonical             user
  x | px  vs x | x'                  4.52706607e+00   4.52706607e+00


  x | delta                          3.88247680e-02   3.88247680e-02
  px | delta  vs  x' | delta        -1.38881714e-03  -1.38881714e-03
  l | delta  vs  phi | delta         6.26572031e+01   1.67685720e+01
  l | delta^2  vs phi | delta^2     -7.14865999e+01  -3.19453842e+01
  l | px^2  vs  phi | x'^2          -8.63613560e+01  -2.31123724e+01
  px | px delta vs x'|x' delta       2.09884997e+00   2.09884997e+00
  x | px^2 delta vs x|x'^2 delta     3.41851539e+02   3.42000016e+02

  (phi coefficients include the factor 2 pi h / C and the v(delta) terms)

--- 5. Symplecticity

  max |J^T S J - S| of the map Jacobian at z = (1e-3, 1e-3, 1e-3, 1e-3, 0, 1e-3)

  canonical map: 6.74e-06  (zero up to truncation at order 3)
  user map     : 1.28e-01  (not small: (x,x',y,y',phi,delta) is not a canonical set)

Done.


## 16 — BLonD-like longitudinal drift (external formulae)

`(dt, dE)` drift of BLonD compared with element tracking. The formulae
live in this example, not in `src/`.


In [9]:
run_example(joinpath("16_blond_longitudinal", "run.jl"))



  BLonD-like longitudinal drift vs element tracking

  mass used = 1875.61294257 MeV (CODATA deuteron); OptiM header Mass = 1876.544 MeV (ignored)

--- 1. Momentum compaction from the ring map (BLonD alpha_0,1,2)

  Taylor map in 0.0 s



  path_expansion (order 2):  alpha0 = 0.23039874  alpha1 = 0.33854820
  blond_alphas fit:   alpha0 = 0.23039874  alpha1 = 0.33855220  alpha2 = -0.15027723
  BLonD eta (eq.15):         eta0 = -0.53376017  eta1 = 0.73185998  eta2 = -0.15433938
  path_expansion eta:        eta0 = -0.53376017  eta1 = 1.01675589  (TOF identity)
  |alpha0 fit - path| = -1.18e-11  |alpha1 fit - path| = 4.00e-06

--- 2. One-turn magnetic drift: element tracking vs BLonD

  Particles on the closed orbit at fixed delta (x = D delta, ...). No RF.

       delta     Dphi exact     Dphi BLonD    Dphi simple    rel BLonD   rel simple


  -2.0000e-03  -3.366513e-02  -3.366518e-02  -3.351150e-02     1.75e-06    -4.56e-03
  -1.0000e-03  -1.680051e-02  -1.680055e-02  -1.676216e-02     1.98e-06    -2.28e-03
  -5.0000e-04  -8.392258e-03  -8.392275e-03  -8.382683e-03     2.05e-06    -1.14e-03
  5.0000e-04   8.376286e-03   8.376304e-03   8.385887e-03     2.11e-06     1.15e-03
  1.0000e-03   1.673663e-02   1.673666e-02   1.677498e-02     2.08e-06     2.29e-03
  2.0000e-03   3.340958e-02   3.340964e-02   3.356276e-02     1.92e-06     4.58e-03

--- 3. With RF: thin kick + BLonD exact drift vs full tracking (32 turns)

  Stationary bucket phi_s = 180 deg, V = 0.2 MV.
  BLonD uses an instantaneous kick; tracking uses the finite cavity.

    turn    phi track    phi BLonD      d track      d BLonD


       1      0.01671      0.01674  9.97413e-04  1.00000e-03
       4      0.06574      0.06585  9.50519e-04  9.60568e-04
       8      0.12462      0.12481  8.01848e-04  8.20557e-04
      16      0.19867      0.19881  2.79023e-04  3.07730e-04
      32      0.10750      0.10682 -8.47909e-04 -8.33612e-04

  max |phi_track - phi_BLonD| over 32 turns = 6.733e-04 rad
  max |delta_track - delta_BLonD|           = 2.939e-05
  (small residual expected: finite cavity length + transverse coupling vs thin kick + 1D drift)

Done.


## 17 — Lie product of two factors

`concatenate(Fa, Fb)` versus the Lie series of a pair of elements.
Fixes the sign of the BCH term.

Block: Lie algebra (product of Dragt–Finn factors).


In [10]:
run_example(joinpath("17_lie_product", "run.jl"))



  Lie product of Dragt-Finn factors to degree 4


--- 1. Single element: Lie series vs Dragt-Finn reconstruction

  elem          recon         |f3|         |f4|
  D         0.000e+00    1.000e+00    1.000e+00


  QF        5.551e-17    2.032e-01    2.049e-01
  QD        5.551e-17    2.032e-01    2.049e-01
  SX        2.220e-16    8.000e-01    1.000e-01
  B         2.220e-16    7.493e-01    7.503e-01



--- 2. Pair of elements: concatenate vs composed Lie series

  pair                |Δf3|        |Δf4|    |Δf4 opp|         |ΔR|       |Δmap|


  D+QF            2.220e-16    2.220e-16    5.653e-02    0.000e+00    4.441e-16


  QF+QD           5.551e-17    5.551e-17    1.030e-05    2.220e-16    3.331e-16
  QF+SX           2.220e-16    1.110e-16    6.735e-01    1.110e-16    5.551e-16


  SX+D            2.776e-17    4.510e-17    3.200e+00    0.000e+00    4.441e-16
  B+QF            1.110e-16    1.110e-16    4.191e-02    1.388e-17    2.220e-16


  QF+B            1.110e-16    1.110e-16    3.713e-02    1.388e-17    4.441e-16
  SX+SX           5.551e-17    1.110e-16    2.560e-01    0.000e+00    4.441e-16

--- 3. Three elements (fold)

  |Δf3| = 3.331e-16  |Δf4| = 1.110e-16  |Δmap| = 5.551e-16



Done.


## 18 — Normal form on generators

One-turn Lie factor with thin RF, then the homological equation on
`f3`, `f4`. Torus average of laboratory `δ`.

Block: normal form.


In [11]:
run_example(joinpath("18_lie_normal_form", "run.jl"))



  One-turn Lie factor and normal form

  one-turn factor in 0.15 s; |f3|=8.636e+01  |f4|=3.533e+02


  |const of factor map| = 0.000e+00 (origin must stay)



  NF in 2.07 s
  Q = (0.462452, 0.462016, 0.012960)
  |A^{-1} R A - rot| = 5.551e-15   |A'SA - S| = 8.517e-16

  |f3| before = 2.566e-01   after F3 = 4.441e-16
  |f4 - h4| after F4 = 9.215e-14
  d (torus, from T) = 1.46652439e-01  2.23289406e-01  2.47706390e-03
  d (grid check)    = 1.46652439e-01  2.23289406e-01  2.47706390e-03
  |d - d_grid|      = 6.735e-16

  d (TOF formula)   = 1.46652568e-01  2.23289712e-01  2.47706416e-03
  rel d − formula   = -8.765e-07  -1.372e-06  -1.065e-07

  cubic homological residual |f3+F3-F3∘r| = 4.441e-16

Done.


## Done

Every numbered example has been included. Plots and GIFs are under
`examples/NN_name/output/`.


In [12]:
println("finished = ", now())


finished = 2026-10-30T12:16:02.724
